In [3]:
!pip install pandas ipywidgets textblob scikit-learn tqdm numpy openpyxl matplotlib sentence-transformers cupy xgboost seaborn flask streamlit joblib textblob google-generativeai python-dotenv emoji

  Using cached cupy-13.5.1.tar.gz (3.5 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached google_generativeai-0.8.5-py3-none-any.whl.metadata (3.9 kB)
  Using cached emoji-2.14.1-py3-none-any.whl.metadata (5.7 kB)
  Using cached fastrlock-0.8.3-cp311-cp311-win_amd64.whl.metadata (7.9 kB)
  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached google_api_core-2.25.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached google_api_python_client-2.177.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached google_auth-2.40.3-py2.py3-none-any.whl.metadata (6.2 kB)
  Using cached proto_plus-1.26.1-

  error: subprocess-exited-with-error
  
  × Building wheel for cupy (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [78 lines of output]
      Clearing directory: C:\Users\swaya\AppData\Local\Temp\pip-install-cstpdo1g\cupy_2d47ecb2b12840faa345c2909b2af2a9\cupy\.data
      Generating CUPY_CACHE_KEY from header files...
      CUPY_CACHE_KEY (1729 files matching C:\Users\swaya\AppData\Local\Temp\pip-install-cstpdo1g\cupy_2d47ecb2b12840faa345c2909b2af2a9\cupy\_core\include\**): 62426478e3e7017e0abfdd71b0667fdffa294302
      **************************************************
      *** WARNING: nvcc path != CUDA_PATH
      *** WARNING: nvcc path: None
      *** WARNING: CUDA_PATH: C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9
      **************************************************
      Looking for NVTX: C:\Program Files\NVIDIA Corporation\Nsight Systems *\target-windows-x64\nvtx
      Using NVTX at: C:\Program Files\NVIDIA Corporation\Nsight Systems 2025.1.3\t

In [1]:
import pandas as pd

df = pd.read_excel("behaviour_simulation_train.xlsx", parse_dates = ['date']) #add path to the dataset

# Data cleaning
df['content'] = df['content'].astype(str).str.strip()
df['username'] = df['username'].astype(str).str.strip()
df['media'] = df['media'].astype(str).str.strip()

# Feature engineering
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.day_name()
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

media_dict = {'P':'Photo', 'V':'Video', 'G':'Gif'}
df['media_type'] = df['media'].str[1].map(media_dict)

df['word_count'] = df['content'].str.split().str.len()
df['char_count'] = df['content'].str.len()


In [5]:
!pip install emoji

  Using cached emoji-2.14.1-py3-none-any.whl.metadata (5.7 kB)
Using cached emoji-2.14.1-py3-none-any.whl (590 kB)


In [5]:
import emoji

def count_emojis(text):
    return sum(1 for char in text if char in emoji.EMOJI_DATA)

df['emoji_count'] = df['content'].apply(lambda x: count_emojis(x))
df['hashtag_count'] = df['content'].str.count(r'#\w+')

In [6]:
# Sentiment analysis using TextBlob, adding polarity and subjectivity as features

from textblob import TextBlob
from tqdm.notebook import tqdm

def analyze_sentiments(texts):
    polarities = []
    subjectivities = []
    
    # Use tqdm for progress tracking
    for text in tqdm(texts, desc="Analyzing texts"):
        analysis = TextBlob(text)
        polarities.append(analysis.sentiment.polarity)
        subjectivities.append(analysis.sentiment.subjectivity)
    
    return polarities, subjectivities

df['sentiment_polarity'], df['sentiment_subjectivity'] = analyze_sentiments(df['content'])

Analyzing texts:   0%|          | 0/300000 [00:00<?, ?it/s]

In [7]:
# rmse when predicting average likes for each row
# rmse: 4931.455200077609
from sklearn.metrics import mean_squared_error

import numpy as np

average = df['likes'].mean()

rmse = np.sqrt(mean_squared_error(df['likes'], average * np.ones_like(df['likes'])))
print("RMSE:", rmse)

RMSE: 4931.455200077609


In [8]:
usernames_data = pd.read_csv("usernames_data_x.csv", parse_dates=['createdAt'])
usernames_data.info()
display(usernames_data.isnull().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2249 entries, 0 to 2248
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   userName         2249 non-null   object             
 1   id               2249 non-null   float64            
 2   name             2249 non-null   object             
 3   followers        2249 non-null   int64              
 4   following        2249 non-null   int64              
 5   isBlueVerified   2249 non-null   bool               
 6   verifiedType     604 non-null    object             
 7   statusesCount    2249 non-null   int64              
 8   mediaCount       2249 non-null   int64              
 9   favouritesCount  2249 non-null   int64              
 10  createdAt        2249 non-null   datetime64[ns, UTC]
 11  location         1788 non-null   object             
 12  description      2184 non-null   object             
 13  canDm            2

userName              0
id                    0
name                  0
followers             0
following             0
isBlueVerified        0
verifiedType       1645
statusesCount         0
mediaCount            0
favouritesCount       0
createdAt             0
location            461
description          65
canDm                 0
dtype: int64

In [9]:
if df.shape[1]<30: # so that duplicate we do not merge again by mistake and duplicate columns are made
    # (after merging number of columns will be 30)
    df = pd.merge(df, usernames_data, left_on='username', right_on='userName', how='left')
print(df.info())
display(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 32 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   id_x                    300000 non-null  int64              
 1   date                    300000 non-null  datetime64[ns]     
 2   likes                   300000 non-null  int64              
 3   content                 300000 non-null  object             
 4   username                300000 non-null  object             
 5   media                   300000 non-null  object             
 6   inferred company        300000 non-null  object             
 7   hour                    300000 non-null  int32              
 8   day_of_week             300000 non-null  object             
 9   month                   300000 non-null  int32              
 10  year                    300000 non-null  int32              
 11  media_type              30

id_x                           0
date                           0
likes                          0
content                        0
username                       0
media                          0
inferred company               0
hour                           0
day_of_week                    0
month                          0
year                           0
media_type                     0
word_count                     0
char_count                     0
emoji_count                    0
hashtag_count                  0
sentiment_polarity             0
sentiment_subjectivity         0
userName                   11656
id_y                       11656
name                       11656
followers                  11656
following                  11656
isBlueVerified             11656
verifiedType              178564
statusesCount              11656
mediaCount                 11656
favouritesCount            11656
createdAt                  11656
location                   68585
descriptio

In [10]:
df['username_avg_likes'] = df.groupby('username')['likes'].transform('mean')
df['company_avg_likes'] = df.groupby('inferred company')['likes'].transform('mean')

In [30]:
df.info()
display(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   id_x                    300000 non-null  int64              
 1   date                    300000 non-null  datetime64[ns]     
 2   likes                   300000 non-null  int64              
 3   content                 300000 non-null  object             
 4   username                300000 non-null  object             
 5   media                   300000 non-null  object             
 6   inferred company        300000 non-null  object             
 7   hour                    300000 non-null  int32              
 8   day_of_week             300000 non-null  object             
 9   month                   300000 non-null  int32              
 10  year                    300000 non-null  int32              
 11  media_type              30

id_x                           0
date                           0
likes                          0
content                        0
username                       0
media                          0
inferred company               0
hour                           0
day_of_week                    0
month                          0
year                           0
media_type                     0
word_count                     0
char_count                     0
sentiment_polarity             0
sentiment_subjectivity         0
userName                   11656
id_y                       11656
name                       11656
followers                  11656
following                  11656
isBlueVerified             11656
verifiedType              178564
statusesCount              11656
mediaCount                 11656
favouritesCount            11656
createdAt                  11656
location                   68585
description                15284
canDm                      11656
username_a

In [11]:
display(df.head())

,id_x,date,likes,content,username,media,inferred company,hour,day_of_week,month,...,verifiedType,statusesCount,mediaCount,favouritesCount,createdAt,location,description,canDm,username_avg_likes,company_avg_likes
0,1,2020-12-12 00:47:00,1,"Spend your weekend morning with a Ham, Egg, an...",TimHortonsPH,[Photo(previewUrl='https://pbs.twimg.com/media...,tim hortons,0,Saturday,12,...,NaN,931.0,687.0,954.0,2016-11-04 05:57:45+00:00,Republic of the Philippines,Welcome to the official account of Tim Hortons...,False,3.670270,160.179678
1,2,2018-06-30 10:04:20,2750,Watch rapper <mention> freestyle for over an H...,IndyMusic,[Photo(previewUrl='https://pbs.twimg.com/media...,independent,10,Saturday,6,...,NaN,31733.0,4853.0,104.0,2008-11-11 12:43:57+00:00,London,"Music news, reviews, and feature stories from ...",False,441.894737,49.258918
2,3,2020-09-29 19:47:28,57,Canadian Armenian community demands ban on mil...,CBCCanada,[Photo(previewUrl='https://pbs.twimg.com/media...,cbc,19,Tuesday,9,...,NaN,93717.0,53557.0,11.0,2009-01-14 03:42:38+00:00,Canada,"Canadian news and features, and the Politics B...",False,95.671698,204.387701
3,4,2020-10-01 11:40:09,152,"1st in Europe to be devastated by COVID-19, It...",MKWilliamsRome,[Photo(previewUrl='https://pbs.twimg.com/media...,williams,11,Thursday,10,...,NaN,5739.0,1294.0,10180.0,2012-08-13 10:53:48+00:00,NaN,"Report on radio & TV & write on Italy, Europe ...",False,256.151515,2093.835052
4,5,2018-10-19 14:30:46,41,Congratulations to Pauletha Butts of <mention>...,BGISD,[Photo(previewUrl='https://pbs.twimg.com/media...,independent,14,Friday,10,...,NaN,7397.0,2347.0,5311.0,2009-02-26 21:06:29+00:00,"Bowling Green, Kentucky","Every day, accomplishments of students and emp...",False,63.403226,49.258918


In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#import seaborn as sns

# computing the correlation matrix
corr_matrix = df[['likes', 
    'followers', 
    'following', 
    'statusesCount',
    'mediaCount', 
    'favouritesCount', 
    'sentiment_polarity', 
    'sentiment_subjectivity',
    'char_count',
    'word_count',
    'username_avg_likes',
    'company_avg_likes',
    'emoji_count',
    'hashtag_count']].corr()

display(corr_matrix)

# plotting heatmap
# plt.figure(figsize=(8, 6))
# sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt=".2f")
# plt.title('Correlation Matrix Heatmap')
# plt.tight_layout()
# plt.show()


,likes,followers,following,statusesCount,mediaCount,favouritesCount,sentiment_polarity,sentiment_subjectivity,char_count,word_count,username_avg_likes,company_avg_likes,emoji_count,hashtag_count
likes,1.000000,0.114841,-0.004574,-0.020911,-0.005188,0.030032,0.002094,0.007150,-0.015755,-0.003997,0.447372,0.367532,0.036158,-0.002787
followers,0.114841,1.000000,-0.038959,0.184327,0.374044,-0.085150,-0.066777,0.000332,0.102609,0.112605,0.256715,0.287338,-0.073130,-0.166777
following,-0.004574,-0.038959,1.000000,0.065142,-0.100997,0.112062,0.023166,0.029696,0.035301,0.036914,-0.010226,-0.007884,0.054217,0.040726
statusesCount,-0.020911,0.184327,0.065142,1.000000,0.730172,-0.099159,-0.132805,-0.125445,-0.191989,-0.193390,-0.046745,-0.049138,-0.105030,-0.235161
mediaCount,-0.005188,0.374044,-0.100997,0.730172,1.000000,-0.174254,-0.191893,-0.182301,-0.233076,-0.242162,-0.011596,-0.013236,-0.150466,-0.314259
favouritesCount,0.030032,-0.085150,0.112062,-0.099159,-0.174254,1.000000,0.017523,0.021523,-0.065749,-0.032158,0.067134,0.116662,0.063183,-0.064780
sentiment_polarity,0.002094,-0.066777,0.023166,-0.132805,-0.191893,0.017523,1.000000,0.468945,0.134472,0.138816,-0.007275,0.003001,0.059095,0.113603
sentiment_subjectivity,0.007150,0.000332,0.029696,-0.125445,-0.182301,0.021523,0.468945,1.000000,0.261735,0.273016,0.010336,0.009129,0.034796,0.086073
char_count,-0.015755,0.102609,0.035301,-0.191989,-0.233076,-0.065749,0.134472,0.261735,1.000000,0.971636,-0.038806,-0.045682,0.010951,0.305030
word_count,-0.003997,0.112605,0.036914,-0.193390,-0.242162,-0.032158,0.138816,0.273016,0.971636,1.000000,-0.014384,-0.021134,0.036770,0.221868


In [14]:
def get_features(X_train, X_test, numerial_features=[], categorical_features=[], avg_likes=[], bert_embeddings=None):
    X_train_final = np.hstack([
        X_train[i].values.reshape(-1, 1) for i in numerial_features
    ])
    X_test_final = np.hstack([
        X_test[i].values.reshape(-1, 1) for i in numerial_features
    ])

    from sklearn.preprocessing import OneHotEncoder
    
    encoders = {}  # To store encoders for each categorical feature
    for i in categorical_features:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        encoder.fit(X_train[[i]])
        encoded_train = encoder.transform(X_train[[i]])
        encoded_test = encoder.transform(X_test[[i]])
        X_train_final = np.hstack([X_train_final, encoded_train])
        X_test_final = np.hstack([X_test_final, encoded_test])
        encoders[i] = encoder  # Store the encoder for later use
        import sys
    
    for i in avg_likes:
        avg_likes_train = X_train.groupby(i)['likes'].mean().reindex(X_train[i])
        avg_likes_test = X_test.groupby(i)['likes'].mean().reindex(X_test[i])
        X_train_final = np.hstack([X_train_final, avg_likes_train.values.reshape(-1, 1)])
        X_test_final = np.hstack([X_test_final, avg_likes_test.values.reshape(-1, 1)])
    print('features obtained')
    return X_train_final, X_test_final, encoders

In [15]:
def random_forest_regressor(X_train_final, y_train, X_test_final, y_test, n_estimators=100):
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_squared_error
    import numpy as np
    
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train_final, y_train)
    
    preds = rf_model.predict(X_test_final)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print("RMSE:", rmse)
    return rf_model

In [16]:
def xgboost_regressor(X_train_final, y_train, X_test_final, y_test, early_stopping_rounds=10, n_estimators=1000):
    from xgboost import XGBRegressor
    from sklearn.metrics import mean_squared_error
    
    xgb_model = XGBRegressor(
        device='cuda',
        early_stopping_rounds=early_stopping_rounds,  # to prevent overfitting
        eval_metric='rmse',       
        n_estimators=1000          
    )
    xgb_model.fit(
        X_train_final, y_train,
        eval_set=[(X_test_final, y_test)],
        verbose=True
    )
    preds = xgb_model.predict(X_test_final)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print("RMSE:", rmse)
    return xgb_model

In [17]:
# Splitting the dataset into training and testing sets
from sklearn.model_selection import train_test_split
import numpy as np

X = df
y = df['likes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# rmse = 4059.8200699045765

X_train_final, X_test_final, encoders = get_features(
    X_train, X_test,
    numerial_features=['char_count', 'sentiment_polarity', 'emoji_count', 'hashtag_count'],
    categorical_features=['media_type', 'inferred company', 'username']
)
xg_model = xgboost_regressor(X_train_final, y_train, X_test_final, y_test)
import joblib
joblib.dump(xg_model, 'model_1_xg.pkl')

features obtained
[0]	validation_0-rmse:4544.89843
[1]	validation_0-rmse:4412.84302
[2]	validation_0-rmse:4370.20807
[3]	validation_0-rmse:4254.94829
[4]	validation_0-rmse:4236.71899
[5]	validation_0-rmse:4221.16636
[6]	validation_0-rmse:4151.84740
[7]	validation_0-rmse:4148.59078
[8]	validation_0-rmse:4143.12054
[9]	validation_0-rmse:4118.64228
[10]	validation_0-rmse:4111.54815
[11]	validation_0-rmse:4107.10485
[12]	validation_0-rmse:4115.08432
[13]	validation_0-rmse:4110.98414
[14]	validation_0-rmse:4119.05668
[15]	validation_0-rmse:4122.84389
[16]	validation_0-rmse:4116.14755
[17]	validation_0-rmse:4106.60542
[18]	validation_0-rmse:4104.66015
[19]	validation_0-rmse:4103.48976
[20]	validation_0-rmse:4070.06343
[21]	validation_0-rmse:4068.46054
[22]	validation_0-rmse:4070.21454
[23]	validation_0-rmse:4059.82013
[24]	validation_0-rmse:4074.72556
[25]	validation_0-rmse:4070.19550
[26]	validation_0-rmse:4067.68653
[27]	validation_0-rmse:4064.96783
[28]	validation_0-rmse:4082.07645
[29]	v

c:\Users\swaya\OneDrive\Documents\GitHub\AI_and_Dev_Swayam_Pandey_CSoT\.venv\Lib\site-packages\xgboost\core.py:729: UserWarning: [00:27:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


RMSE: 4059.8200699045765


['model_0_xg.pkl']

In [26]:
# RMSE: 4050.0509873333694

X_train_final, X_test_final, encoders = get_features(
    X_train, X_test,
    numerial_features=['char_count', 'sentiment_polarity', 'emoji_count', 'hashtag_count', 'followers', 'following'],
    categorical_features=['media_type'],
)
xg_model = xgboost_regressor(X_train_final, y_train, X_test_final, y_test)
import joblib
joblib.dump(xg_model, 'model_2_xg.pkl')
for i in encoders:
    joblib.dump(encoders[i], f'encoder_{i}_2_xg.pkl')

features obtained
[0]	validation_0-rmse:4684.85254
[1]	validation_0-rmse:4414.44616
[2]	validation_0-rmse:4246.47790
[3]	validation_0-rmse:4167.79314
[4]	validation_0-rmse:4112.14107
[5]	validation_0-rmse:4092.36410
[6]	validation_0-rmse:4085.37174
[7]	validation_0-rmse:4079.53034
[8]	validation_0-rmse:4064.19453
[9]	validation_0-rmse:4075.45899
[10]	validation_0-rmse:4072.56416
[11]	validation_0-rmse:4077.20844
[12]	validation_0-rmse:4072.43461
[13]	validation_0-rmse:4069.67871
[14]	validation_0-rmse:4067.98524
[15]	validation_0-rmse:4069.53656
[16]	validation_0-rmse:4065.25084
[17]	validation_0-rmse:4054.98847
[18]	validation_0-rmse:4052.49727
[19]	validation_0-rmse:4056.12529
[20]	validation_0-rmse:4058.66548
[21]	validation_0-rmse:4056.69763
[22]	validation_0-rmse:4055.82210
[23]	validation_0-rmse:4055.62721
[24]	validation_0-rmse:4050.05104
[25]	validation_0-rmse:4051.44191
[26]	validation_0-rmse:4054.27273
[27]	validation_0-rmse:4079.71581
[28]	validation_0-rmse:4081.22899
[29]	v

with hour rmse turned out to be higher (~4330)<br>
with sentiment_subjectivity, rmse turned out to be higher (4276)

In [28]:
# RMSE: 4332.59771449873, will take 3-4 hours
X_train_final, X_test_final = get_features(
    X_train, X_test,
    numerial_features=['char_count', 'sentiment_polarity', 'emoji_count', 'hashtag_count', 'followers', 'following'],
    categorical_features=['media_type', 'inferred company', 'username'],
    avg_likes=['username', 'inferred company']
)
rf_model = random_forest_regressor(X_train_final, y_train, X_test_final, y_test)

KeyboardInterrupt: 

In [29]:
bert_embedding = pd.read_csv('embeddings.csv')

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(df['content'].tolist(), show_progress_bar=True, convert_to_numpy=True)
print(embeddings.shape)
print(embeddings)

In [30]:
import torch
import torch.nn as nn

class Like_Predictor(nn.Module):
    def __init__(self, embedding_dimension, numerical_features_count, categorical_features_dict):
        super().__init__()
        # Dense layer for the CLS embedding
        self.bert_embeddings_layer = nn.Sequential(
            nn.Linear(embedding_dimension, 128),
            nn.ReLU()
        )
        # Embedding layers for categorical features
        self.categorical_embeddings_layer = nn.ModuleDict({
            name: nn.Embedding(num_classes, emb_dim)
            for name, (num_classes, emb_dim) in categorical_features_dict.items()
        })
        # Dense for structured numerical features
        self.numerical_features_layer = nn.Sequential(
            nn.Linear(numerical_features_count, 32),
            nn.ReLU()
        )
        # Final classifier
        total_dim = 128 + 32 + sum(emb_dim for _, (_, emb_dim) in categorical_features_dict.items())
        self.final_nn = nn.Sequential(
            nn.Linear(total_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, bert_embedding, numerical_features, categorical_features):
    
        bert_embeds_layer = self.bert_embeddings_layer(bert_embedding)  # (batch, 128)
        num_layer = self.numerical_features_layer(numerical_features)  # (batch, 32)
        # Process categorical features
        cat_embeds = [
            self.categorical_embeddings_layer[name](categorical_features[name])  # (batch, emb_dim)
            for name in self.categorical_embeddings_layer
        ]
        # Concatenate categorical embeddings
        cat_embeds_layer = torch.cat(cat_embeds, dim=1) if cat_embeds else None

        if cat_embeds_layer is not None:
            all_features_layer = torch.cat([bert_embeds_layer, num_layer, cat_embeds_layer], dim=1)
        else:
            all_features_layer = torch.cat([bert_embeds_layer, num_layer], dim=1)
        out = self.final_nn(all_features_layer)
        return out.squeeze(1)  # (batch,)




In [32]:
numerical_features = pd.concat = df[['char_count', 'emoji_count', 'hashtag_count', 'followers', 'following', 'statusesCount', 'favouritesCount', 'mediaCount']]


In [33]:
# changing where 'verifiedType' is null to 'None'
df['verifiedType'] = df['verifiedType'].fillna('None')

In [34]:
categorical_features = {
    'media_type': df['media_type'].astype('category').cat.codes,
    'hour': df['hour'],
    'verifiedType': usernames_data['verifiedType'].astype('category').cat.codes
}

In [40]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Configuration
embedding_dimension = 384
numerical_features_count = 8 # char_count, followers, following, statusesCount, favouritesCount, mediaCount, emoji_count, hashtag_count
categorical_features_dict = {
    'media_type': (3, 8),     # 4 types, 8-dim embedding
    'hour': (24, 4),     # 24 hours, 4-dim embedding
    'verifiedType': (3, 4)  # 3 types, 4-dim embedding
}

# Model
model = Like_Predictor(embedding_dimension, numerical_features_count, categorical_features_dict).to(device)

# Dummy batch data

batch_size = 30

targets = df['likes']

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
model.train()
# Training step
for i in range(0, len(bert_embedding), batch_size):
    bert_embedding_batch = torch.tensor(bert_embedding[i:i + batch_size].values, dtype=torch.float32).to(device)
    numerical_features_batch = torch.tensor(numerical_features[i:i + batch_size].values, dtype=torch.float32).to(device)
    categorical_features_batch = {
        name: torch.tensor(categorical_features[name][i:i + batch_size].values, dtype=torch.long).to(device)
        for name in categorical_features
    }
    targets_batch = torch.tensor(targets[i:i + batch_size].values, dtype=torch.float32).to(device)

    optimizer.zero_grad()
    outputs = model(bert_embedding_batch, numerical_features_batch, categorical_features_batch)
    loss = criterion(outputs, targets_batch)
    loss.backward()
    optimizer.step()
    print(f"Batch {i // batch_size + 1}, Loss: {loss.item()}")
# Save the model
joblib.dump(model, 'like_predictor_nn.pkl')


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
